# AI Fix Verifier: demo

This notebook checks whether a security fix **removes a vulnerability without adding a new one**.
It scans the code before and after the fix with [Bandit](https://bandit.readthedocs.io/), compares the findings, and gives a verdict: **PASS**, **REJECT** or **REVIEW**.

**How to use:** click **Runtime → Run all**. It takes about two minutes and needs no setup. If an output is collapsed, click *Show hidden output*.

Repository: https://github.com/Sarsa-25/ai-fix-verifier

## What this project found

Comparing a "before" scan with an "after" scan sounds simple, but my first version got three things wrong. Each one was found by a test case and fixed:

| Problem | What went wrong | Fix | Case |
|---|---|---|---|
| A comment on the vulnerable line | the line's text changed, so one unfixed bug looked like "resolved + introduced" | strip comments before comparing (`tokenize`) | C03 |
| Two identical vulnerable lines, one fixed | the partial fix was invisible | count findings instead of only checking if they exist (`Counter`) | C04 |
| `# nosec` added to the vulnerable line | Bandit skipped the line, so a fake fix passed | run Bandit with `--ignore-nosec` | C19 |

Three limits remain, and the test set measures them: a safe `subprocess` fix is still rejected (C05), an old unrelated finding blocks a correct fix (C12, also seen in a real open-source project), and deleting the vulnerable code looks like a fix (C17).

The tool was also tried on real code: a 2017 security fix in OpenStack heat-translator, and a Flask debug-mode vulnerability that I found, fixed and verified in my own project. Details are in the README.

## 1. Setup
Download the project with all test-case branches, and install Bandit.

In [ ]:
!git clone -q https://github.com/Sarsa-25/ai-fix-verifier.git
%cd ai-fix-verifier
!git fetch -q --update-head-ok origin "+refs/heads/*:refs/heads/*"
!pip -q install bandit==1.9.4
!python -m bandit --version | head -1

import os
TOOL = os.path.join(os.getcwd(), "verify.py")   # used later to check fixes in other projects
WORK = os.path.dirname(os.getcwd())             # other projects are cloned next to this one

## 2. One example: a fix that adds a new vulnerability

The fix below removes the SQL injection, but adds a hardcoded password. First, the code change:

In [ ]:
!git --no-pager diff main fix/sqli-with-secret -- vulnerable_app.py

Now the tool checks the fix. Expected: the SQL injection (B608) is **resolved**, the password (B105) is **introduced**, so the verdict is **REJECT**.

In [ ]:
!python verify.py main fix/sqli-with-secret vulnerable_app.py

## 3. The full test set: 19 cases

Each case is a pair of Git branches (before / after). The cases cover six scenario types: successful fix, incomplete fix, new vulnerability introduced, unrelated pre-existing finding, multiple findings, and ambiguous comparison.

"Expected" is a human judgement of each fix, set before running the tool.

In [ ]:
!python run_cases.py

In [ ]:
import glob
from IPython.display import Markdown, display

latest = sorted(glob.glob("outputs/*_summary.md"))[-1]
display(Markdown(open(latest, encoding="utf-8").read()))

## 4. Look at any case

Change `CASE` and run the cell to see the code change of that case (C05 to C19).

In [ ]:
CASE = "C19"   # try C05, C12, C17 (the three mismatches)

n = CASE.lower()
!git --no-pager diff case/{CASE}-before case/{CASE}-after -- cases/{n}.py

## 5. Real code, part 1: a real open-source fix

The test cases above are small and hand-built. Here the tool checks a real security fix from 2017 in **OpenStack heat-translator**, written by that project's own developers. The commit replaced the unsafe `yaml.load` with `yaml.safe_load` in three files.

In [ ]:
%cd {WORK}
!git clone -q https://github.com/openstack/heat-translator.git
%cd heat-translator

FIX = "3cf56881413"
!git --no-pager log -1 --format="%h  %ad  %s" --date=short {FIX}
!git --no-pager diff --stat {FIX}~1 {FIX}

`{FIX}~1` means "the commit just before the fix". The tool runs once for each changed file.

Expected: all three files resolve B506. `utils.py` is still **REJECT**, because an older finding that has nothing to do with the fix (B113, an HTTP request without a timeout) stays in the file. That is limitation C12, now on real code.

In [ ]:
for f in ["translator/common/utils.py",
          "translator/hot/tosca/tosca_policies_scaling.py",
          "translator/shell.py"]:
    print("=" * 70)
    print(f)
    !python {TOOL} {FIX}~1 {FIX} {f} | sed -n '/Counts/,/Verdict/p'

## 6. Real code, part 2: find, fix and verify in my own project

**job-tracker** is a small Flask app I built earlier. This section follows the full path a developer would take: scan the project, find a vulnerability, fix it, and check the fix.

**Step 1: find.** Bandit scans `app.py` as it was before the fix.

In [ ]:
%cd {WORK}
!git clone -q https://github.com/Sarsa-25/job-tracker.git
%cd job-tracker

FIX = "9448f54"
!git show {FIX}~1:app.py > /tmp/app_before.py
!python -m bandit -q /tmp/app_before.py

Bandit finds **B201**: `app.run(debug=True)`, severity HIGH. Debug mode turns on the Werkzeug debugger, which lets anyone who can reach the app run code on the server.

**Step 2: fix.** This is the change I made: debug mode now stays off unless `FLASK_DEBUG=1` is set.

In [ ]:
!git --no-pager diff {FIX}~1 {FIX} -- app.py

**Step 3: verify.** Expected: B201 resolved, nothing new, verdict **PASS**.

In [ ]:
!python {TOOL} {FIX}~1 {FIX} app.py

## 7. Try it on any fix

Pick any public Python repository on GitHub, a commit that fixes something, and a file it changed. Then run the cell. The values below are an example you can replace.

In [ ]:
REPO   = "https://github.com/openstack/heat-translator.git"
COMMIT = "3cf56881413"
FILE   = "translator/shell.py"

import os
name = REPO.rstrip("/").split("/")[-1].removesuffix(".git")
%cd {WORK}
if not os.path.isdir(name):
    !git clone -q {REPO}
%cd {name}
!python {TOOL} {COMMIT}~1 {COMMIT} {FILE}

## Notes

- 16 of 19 verdicts match. The three mismatches are known limitations: C05 (Bandit flags every `subprocess` use, even a safe one), C12 (the strict policy counts an unrelated finding that was already there), C17 (deleting the vulnerable code looks like a fix to a static scanner).
- The fixes in this test set were written by hand with help from a chat assistant, not by an AI coding tool.
- More detail and all limitations are in the README.